# 02 シナリオと想定QA（ハンズオン1: Genie Agent）

ローコードの **Genie Agent** で、品質保証部の分析エージェントを作ります。修理受付のデータ（テーブル）と社内文書（PDF・PPTX）を横断して、**不具合の原因と対策まで答えるエージェント（RAG）を、コードを書かずに作れること**を確かめます。

このノートブックは読むためのものです。

| # | 内容 |
|---|---|
| 1 | 想定シナリオ |
| 2 | 使うデータ |
| 3 | ハンズオンの流れと事前準備 |
| 4 | Step 2: Genie Agent を作る（**API**: `01_Genie Agent の作成（API）` を実行。画面で作る手順も記載） |
| 5 | Step 3: データを確認する |
| 6 | Step 4: Agent モードで分析する（想定回答・検証ポイント・Volume あり／なしの比較） |
| 7 | まとめ |

## 1. 想定シナリオ

### 会社
家電メーカー「**ブリックス電機**」（架空）。炊飯器・電気ケトル・電子レンジ・掃除機などの生活家電を製造し、全国の家電量販店と EC で販売しています。修理の受付はコールセンター（オペレーター約50人）が電話と Web で行い、出張修理・持込修理・製品交換を手配しています。

### 部門: 品質保証部（数人のチーム）
| | |
|---|---|
| **業務** | 修理の傾向を監視し、不具合の原因を調べて、対策（無償点検・設計変更）を決める |
| **課題** | ・修理受付のデータと、会議資料・技術情報などの文書がバラバラで、原因の特定に時間がかかる<br>・分析はデータ担当に頼むため、待ち時間も長い |
| **実現したいこと** | 「修理データと社内文書を横断して、不具合の原因を自分たちで素早く突き止め、対策を決めたい」 |

### このハンズオンで作るもの
| | |
|---|---|
| 作るもの | ドメインに特化した、チームで使うエージェント（品質保証部の分析エージェント） |
| 作り方 | Genie Agent（ローコード）。Agent モードで、SQL の集計と文書の検索を組み合わせて複数のステップで分析する |
| データ | 構造化データ（テーブル4つ）＋非構造化データ（PDF・PPTX 34ファイル） |
| 伝えたいこと | **ローコードでも本格的なエージェントが作れる**（根拠の SQL と文書を示すので、答えを確かめられる） |

## 2. 使うデータ

対象期間は **2025年4月〜2026年9月（18か月）** です。データの最新月は2026年9月です。

### 2-1. テーブル（Unity Catalog、列コメント付き）

| テーブル | 行数 | 内容 | 主な列 |
|---|---|---|---|
| `products` | 10 | 製品 | `model_no`（型番）, `product_name`, `category`, `warranty_months`（保証期間） |
| `production_lots` | 783 | 製造ロット | `lot_no`, `model_no`, `factory`（新潟・タイ・蘇州）, `production_month`, `quantity`（製造台数） |
| `sales_monthly` | 1,080 | 月次の販売台数（型番×地域） | `sales_month`, `model_no`, `region`, `units_sold` |
| `repair_requests` | 11,884 | 修理受付 | `request_id`, `received_at`, `channel`（電話・Web）, `model_no`, `serial_no`, `lot_no`, `purchase_date`, `symptom_category`, `result_category`, `warranty_status`, `service_cost_jpy`, `operator_memo` |

**テーブルのつながり**（主キー・外部キーを設定済み）

```
products (model_no) ─┬─< production_lots (lot_no, model_no) ─< repair_requests (lot_no)
                     ├─< sales_monthly   (model_no)
                     └─< repair_requests (model_no)
```

- 型番（`model_no`）とロット番号（`lot_no`）で、製品・製造・販売・修理受付がつながります。
- ロット番号は「型番の略号-製造年月+ロット記号」です（例: `RC5-2604B` = RC-500 を2026年4月に製造した B ロット）。製造番号（`serial_no`）の先頭9文字がロット番号です。
- `result_category`（対応結果）: 持込修理／出張修理／製品交換／**修理不要**（電話・メールの案内で解決）／**異常なし**（修理を手配したが、点検の結果、故障ではなかった）
- ⚠️ `operator_memo`（対応メモ）には、お客様の氏名・電話番号・住所がそのまま書き込まれています（ダミーデータ）。

**販売台数あたりの受付率**（販売1,000台あたり）で比べると、季節で販売台数が大きく変わる製品（扇風機・ヒーター・加湿器）も公平に比べられます。

### 2-2. 文書（Volume `docs`）

| フォルダ | 文書 | 形式 | 数 | 内容 |
|---|---|---|---|---|
| `manuals/` | 取扱説明書 | PDF | 10（製品ごとに1本） | 使いかた、お手入れ、トラブル対処の表、エラー表示の一覧 |
| `quality_meetings/` | 品質会議の資料 | PPTX | 18（月1回×18回） | 品質保証部の月次会議。修理受付の概況、部品・工程の変更（変更点管理）、対策の進捗、決定事項 |
| `service_bulletins/` | サービス技術情報 | PDF | 5 | 修理担当者・コールセンター向けの通達。既知の不具合と対処 |
| `policies/` | 保証規定・修理料金表 | PDF | 1 | 保証の条件、修理料金（技術料・部品代・送料）の目安 |

社内の品質会議の資料は、このハンズオン1の分析にだけ使います（お客様への案内の根拠にはしません）。

## 3. ハンズオンの流れと事前準備

| Step | 内容 |
|---|---|
| 1 | 説明（会社・業務・課題、テーブルと文書の紹介） |
| 2 | 作成（`01_Genie Agent の作成（API）` で、テーブル・文書（Volume）・業務の説明（指示）・ベンチマークを設定した Genie Agent を API で作る） |
| 3 | データの確認（Catalog Explorer でテーブルと列コメント、Volume の PDF・PPTX を見る） |
| 4 | 分析（Agent モードで質問を投げ、答えと根拠を確かめる。Volume あり／なしの Agent で答えを比べる） |

### 事前準備（講師・ワークスペース管理者）
| 項目 | 内容 |
|---|---|
| プレビューの有効化（**管理者**） | ワークスペースの［プレビュー］で「**Genie エージェントによるボリューム内のファイル分析**（Analyze Files in Volumes with Genie Agents）」をオンにする（ベータ） |
| データの準備 | `00_セットアップ` を実行する（スキーマ・Volume・テーブルを作り、CSV と文書をコピーする）。続けて `01_Genie Agent の作成（API）` を実行する |
| SQL ウェアハウス | Pro またはサーバーレスの SQL ウェアハウス。参加者に `CAN USE` |
| 参加者の権限 | テーブルへの `SELECT`、`docs` Volume への `READ VOLUME`（Genie Agent は質問した人の権限でデータを読む） |
| 対応リージョン | 文書の分析（コンテンツ検索）は対応リージョンのみ。AWS の東京リージョンや、Azure の東日本リージョンなどは Geo 間処理なしで使える |

## 4. Step 2: Genie Agent を作る

### 4-0. API で作る（ハンズオンの手順）
`01_Genie Agent の作成（API）` を開き、ウィジェットでカタログ名とスキーマ名を指定して［**すべてを実行**］を押します。`[Volumeあり]` と `[Volumeなし]` の2つの Agent が作られます（4-1〜4-4 の設定と、ベンチマーク5問がまとめて入ります）。作ったあと、4-2 の 3. のとおりインデックスの状態を確かめてください。

以下の 4-1〜4-4 は、**画面で作る場合**の手順です。

### 4-1. Genie Agent を作り、テーブルを追加する
1. 左のサイドバーの［**Genie エージェント**］をクリックし、右上の［**新規**］をクリックします。
2. データソースとして、次の **4つのテーブル** を選び、［**作成**］をクリックします。
   - `products` / `production_lots` / `sales_monthly` / `repair_requests`
3. 右上の［**設定する**］を押すと、右側に設定のパネル（［**概要**］［**ソース**］［**指示**］［**例**］のタブ）が開きます。
4. ［**概要**］の「**このエージェントについて**」の鉛筆で、名前（例: `ブリックス電機 品質保証部 分析エージェント`）とデフォルトのウェアハウスを設定し、［**保存**］を押します。

### 4-2. 文書（Volume）を追加する
1. ［**設定する**］→［**ソース**］→［**追加**］→［**ボリューム**］を選びます。
2. Volume `docs` を選び、［**確認**］を押します（Volume 全体が対象になります。フォルダやファイルの一部だけを追加することはできません。`csv` は追加しません）。
3. ソースの一覧で `docs` を開き、コンテンツ検索のステータスが「**同期完了**」、処理済みファイルが **34/34** になっていることを確かめます。

> 💡 Volume を追加すると、**コンテンツ検索**が自動で有効になり、ファイルが解析（`ai_parse_document`）されてインデックスが作られます。最初は数分かかることがあります。ファイルを追加・変更・削除すると、インデックスは自動で同期されます。

### 4-3. 業務の説明（指示）を書く
［**設定する**］→［**指示**］の「**一般的な指示**」に次の文章を貼り付け、［**Save**］を押します。データの意味（受付率の定義、修理が不要だった受付の定義、費用の根拠、個人情報の扱い）を、ここでエージェントに教えます。

```
# あなたの役割
あなたは、家電メーカー「ブリックス電機」の品質保証部の分析エージェントです。修理受付のデータ（テーブル）と社内文書（Volume）を組み合わせて、修理の傾向の監視、不具合の原因の調査、対策（無償点検・設計変更など）とその費用の検討を手伝います。

# データの使い分け
- 件数・率・金額は、必ずテーブルを集計して答える。
- 原因や背景（部品・工程の変更、過去の不具合と対策、使いかたの説明、費用の単価）は、文書（docs Volume）から探す。
  - quality_meetings/: 品質保証部の月次の品質会議の資料。部品・工程の変更（変更点管理）、対策の進捗、決定事項
  - service_bulletins/: サービス技術情報。既知の不具合と、受付時・修理時の対応
  - manuals/: 製品ごとの取扱説明書。使いかた、お手入れ、トラブル対処の表、エラー表示の一覧
  - policies/: 保証規定・修理料金表。保証の条件、技術料・部品代・送料
- 答えには、根拠にしたテーブル（集計の条件）と文書の名前を示す。

# 指標の定義
- 受付率 = 修理受付の件数 ÷ 販売台数 × 1,000（販売1,000台あたり）。製品の比較や増減の判断は、件数ではなく受付率で行う（扇風機・ヒーター・加湿器は季節で販売台数が大きく変わるため）。sales_monthly は地域別なので、型番と月で合計してから割る。
- 「実際には修理が不要だったもの」は、result_category が「修理不要」（電話・メールの案内で解決）と「異常なし」（修理を手配したが、点検の結果、故障ではなかった）の受付。
- ロットごとの不具合の多さは、ロットごとの受付件数 ÷ production_lots.quantity（製造台数）で比べる。
- 費用の目安は、修理料金表の単価（技術料・部品代・送料）と、過去の無償交換の実績（実施率・1台あたりの費用・告知費用）を根拠にする。

# 注意
- operator_memo には、お客様の氏名・電話番号・メールアドレス・住所が含まれる。回答に個人情報を出さず、件数や傾向に要約する。
- 型番が似ている製品（RC-500 と RC-550）を取り違えない。
```

### 4-4. （任意）サンプルの質問
［**設定する**］→［**概要**］の「**よくある質問**」に、次の質問を入れておくと、参加者がクリックだけで質問できます。
- この半年で修理の受付が増えている製品を特定し、原因を分析して。さらに修理対象の製品に関して、保証対象の判断基準と実際に取られた保証対応について教えて。

## 5. Step 3: データを確認する（Catalog Explorer）

1. 左のサイドバーの［**カタログ**］を開き、`00_セットアップ` で指定したカタログ → スキーマ（既定は `bricks_electronics`）を開きます。
2. テーブル `repair_requests` を開き、［**概要**］で列コメントを、［**サンプルデータ**］で中身を見ます。
3. Volume `docs` を開き、`manuals/` の PDF をプレビューします。
   - 例: `manuals/manual_RC-500.pdf`

## 6. Step 4: Agent モードで分析する

**文書についての質問は Agent モードでのみ有効です。** 入力欄の下が［**エージェント**］（既定）になっていることを確かめてから質問します（［チャット］に切り替えると、文書は使われません）。

- Agent は、SQL の集計と文書の検索を組み合わせ、複数のステップで分析します（1つの質問に1〜3分ほどかかります）。
- 回答の番号（`[1]` など）から、**根拠の SQL** と **引用した文書（ページ）** を確かめられます。

> 集計を頼むのではなく、**原因と対策まで考えさせる問い**にしているのがポイントです。見るポイントは、Agent がどのテーブルを集計し、どの文書を読んで結論を出したかです。

**Volume あり／なしで比べる**: `01_Genie Agent の作成（API）` で作った `[Volumeあり]` と `[Volumeなし]`（テーブル4つだけ）の2つの Agent に同じ質問を送り、画面を並べて見比べます。

### 質問「この半年で修理の受付が増えている製品を特定し、原因を分析して。さらに修理対象の製品に関して、保証対象の判断基準と実際に取られた保証対応について教えて。」

**ねらい**: 件数ではなく受付率で比べられるか（季節製品のひっかけ）。テーブルの記録をもとに「どこで」起きているかを絞り込み、文書で「なぜ」を突き止められるか。さらに、保証の判断基準を規定（文書）から示し、実際の保証対応をテーブルで確かめられるか。

**Agent の動き（想定）**
1. 製品ごとに、この半年（2026年4〜9月）と前の半年・前年同期の受付率を比べる（`repair_requests` × `sales_monthly`）
2. RC-500 の月別の推移と症状を見る
3. RC-500 をロット・工場・製造月で分ける（`production_lots` と結合し、ロットの製造台数で割る）
4. 文書で原因を探す: 品質会議の資料の「部品・工程の変更」と「対策の進捗」、RC-500 の取扱説明書
5. 保証規定（`warranty_and_repair_fees.pdf`）で保証の判断基準を調べ、対象ロットの受付を `warranty_status`・`result_category`・`service_cost_jpy` で集計する

**想定回答**

| 型番 | 2025/10〜2026/3 件数 | 受付率 | 2026/4〜9 件数 | 受付率 | 受付率の増減 |
|---|---:|---:|---:|---:|---:|
| **RC-500（IH炊飯器）** | 557 | 9.4 | **1,162** | **19.2** | **+103%** |
| FN-210（扇風機） | 29 | 9.4 | 543 | 8.9 | −5%（件数は約19倍） |
| VC-700（掃除機） | 682 | 12.9 | 591 | 11.6 | −10% |
| KT-200（電気ケトル） | 757 | 9.4 | 478 | 6.7 | −29% |
| そのほか6製品 | | | | | ±5%前後で横ばい |

- **受付が増えているのは RC-500**。受付率は2026年4月 9.9 → 5月 10.2 → 6月 13.1 → 7月 20.4 → 8月 29.2 → 9月 **34.3** と加速している。症状は「**途中で止まる**」が中心（月13件 → 216件）。
- 増えているのは **タイ工場で2026年4〜6月に製造した6ロット**（`RC5-2604A/B`・`RC5-2605A/B`・`RC5-2606A/B`）。製造1,000台あたりの受付は 28〜64件で、ほかのロット（多くて12件程度。2026年製造分は7件以下）の数倍。新潟工場の同じ月のロットや、2026年7月以降のロットは平常。
- 文書から分かる原因:
  - `quality_meeting_2026-03.pptx`（変更点管理）: タイ工場で **温度ヒューズの調達先を A社 から K社 に変更（2026年4月製造分から）**。コスト低減のため。評価は n=30 で合格していた。
  - `quality_meeting_2026-06.pptx`: K社品の納入遅延のため、**2026年7月製造分から A社 の品に戻した**（品質の理由ではない）。
  - `quality_meeting_2026-07〜09.pptx`: 受付の増加を監視。延長コードを想定した案内では改善しない。発煙・発火の報告はない。9月度の中間報告で、**回収品12台のうち9台で温度ヒューズが溶断し、定格（145℃）より低い温度で作動するものがあった**。
  - `manual_RC-500.pdf`: 延長コードを使っていないのに炊飯の途中で電源が切れるときは、内部の安全装置（温度ヒューズなど）が作動した可能性がある。
- **結論**: K社製の温度ヒューズを使った期間（タイ工場の2026年4〜6月製造分）の製品で、温度ヒューズが本来より低い温度で作動し、炊飯の途中で電源が切れている可能性が高い。

**想定回答（後半: 保証）**

保証対象の判断基準（`warranty_and_repair_fees.pdf` の保証規定）:
- **保証期間**: お買い上げ日から1年間（RC-500 は12か月。RC-550 だけ2年間）。
- **無償修理**（第2条）: 保証期間内に、取扱説明書に従った正常な使用で故障した場合。引き取り修理の往復送料・出張修理の出張料も当社負担。
- **初期不良**（第3条）: お買い上げから30日以内は、修理ではなく新しい製品と交換。
- **有償になる場合**（第4条）: 誤使用・改造、落下、天災・異常電圧、消耗品（内釜のコーティング・パッキンなど）、保証書がない場合。
- **無償点検・無償交換**（第6条）: サービス技術情報などで告知した対象製品は、保証期間にかかわらず無償。

実際に取られた保証対応（対象6ロットの受付、テーブルで集計）:

| 対応結果 | 件数（うち「途中で止まる」） | 当社負担の平均 | 内訳 |
|---|---:|---:|---|
| 持込修理 | 502（419） | 7,054円（419件は7,040円） | 技術料4,400円＋温度ヒューズ660円＋往復送料1,980円 |
| 製品交換 | 79（64） | 15,390円 | 初期不良（お買い上げから30日以内）。製品原価の目安（希望小売価格の約45%）＋往復送料 |
| 修理不要・異常なし | 24（3） | 0円・3,630円 | 使いかたの案内、点検のみ |
| **合計** | **605（486）** | 当社負担 **約478万円** | **すべて保証期間内**（2026年4〜6月製造で、まだ1年たっていない） |

- まだ無償点検（第6条）は始まっておらず、受付が来たものを、通常の保証の範囲で修理・交換している段階。品質会議 2026年9月度では、対象の範囲と対策・費用を次回までにまとめると決めている。

**検証ポイント**
- ☐ 件数ではなく **受付率（販売1,000台あたり）** で比べているか（`sales_monthly` を型番×月で合計してから割っているか）
- ☐ **FN-210 を「増えている製品」にしていないか**（件数は夏に約19倍だが、受付率は横ばい＝季節要因）
- ☐ ロットの製造台数（`production_lots.quantity`）で割って、**タイ工場の2026年4〜6月製造分**に絞り込めているか
- ☐ 品質会議の資料の **2026年3月度（変更）と 6月度（元に戻す）** を見つけ、ロットの範囲と結び付けているか
- ☐ 根拠の SQL と、引用した文書（ファイル名・ページ）が示されているか
- ☐ 保証の判断基準を、テーブルの推定（購入日＋12か月）だけでなく、**保証規定（PDF）**から示しているか（30日以内の製品交換、有償になる場合、第6条）
- ☐ 対象ロットの受付が**すべて保証内**で、持込修理と製品交換（初期不良）で対応されたことを、件数と費用で示しているか
- ☐ 平均の費用を、修理料金表の単価と照らし合わせているか（7,040円は温度ヒューズの交換の料金とちょうど一致する）
- ☐ 対象台数を **12,650台**（6ロットの製造台数の合計）と数えているか
- ☐ 回答に、お客様の氏名・電話番号などの個人情報が出ていないか

**比べてみよう: Volume あり／なしの Agent で、同じ質問をした結果**

| 観点 | Volume なし（テーブルだけ） | Volume あり |
|---|---|---|
| 増えている製品・範囲 | RC-500（受付率 9.43→19.15）、タイ工場の2026年4〜6月製造の6ロットまで、SQL だけで突き止める | 同じ |
| 原因 | 「社内文書を参照できる手段がなかったため、部品変更などの詳細な原因までは特定できていない」と断って終わる | 品質会議の資料から、温度ヒューズの調達先の変更（3月度）、元に戻した時期（6月度）、溶断の解析（9月度）を引用して原因を示す |
| 保証の判断基準 | `purchase_date`＋保証期間（12か月）と `warranty_status` の整合（99.7%）から**推定**するだけ | 保証規定（PDF）から、保証期間・無償修理の条件・30日以内の製品交換・有償になる場合を示す |
| 実際の保証対応 | 対象ロットの605件がすべて保証内、持込修理502件・製品交換79件、約478万円（テーブルから出せる） | 同じ数字を、修理料金表の単価（技術料・部品代・送料）と照らし合わせて説明する |
| 次のアクション | 「工程・部品の変更の記録を確認するべき」にとどまる | 無償点検・回収交換の検討と、その費用の規模まで示す |

**テーブルで「何が・どこで・どう対応したか」、文書で「なぜ・どんな決まりで」**が分かります。

**答えを確かめる（実際の回答で見つかったこと）**: Agent の答えは毎回少しずつ変わります。上の検証ポイントで、人が確かめてから判断します。
- Volume ありの回答が、対象を「約1.4万台規模」としていた → `production_lots` の6ロットの合計は **12,650台**。
- Volume ありの回答が、持込修理の平均7,040円を「温度ヒューズ以外の部品交換も含まれる可能性」としていた → 7,040円は、技術料4,400円＋温度ヒューズ660円＋往復送料1,980円と**ちょうど一致**（温度ヒューズの交換）。
- Volume なしの回答は、保証の判断基準を「購入日＋12か月」とだけ説明した → 初期不良の製品交換（30日以内）、有償になる場合、第6条（無償点検）は、保証規定（文書）にしか書かれていない。

**深掘りの質問例**
- 「RC-500 の『途中で止まる』を、工場と製造月で分けて、製造1,000台あたりで見せて」
- 「2026年に、部品や工程の変更は他にもあった？ それぞれ受付への影響はあった？」
- 「VC-700 は前年より受付率が下がっている。何があった？」（2025年のバッテリー不良と無償交換）
- 「対象の6ロットを無償点検にしたら、費用はどれくらい？ 過去の無償交換の実績を使って見積もって」（約6,000万円: 12,650台 × 実施率62.6% × 7,040円＋告知費用 約420万円）

## 7. まとめ

| 観点 | このハンズオンで確かめたこと |
|---|---|
| ローコード | テーブルと Volume を選び、業務の説明（指示）を書くだけで、品質保証部の分析エージェントができた |
| 本格的なエージェント（RAG） | Agent モードが、SQL の集計 → ロットの絞り込み → 文書の検索 → 原因と対策・費用の提案、と複数のステップで分析した |
| 構造化＋非構造化 | テーブルで「何が・どこで」、文書で「なぜ・どうする」。片方だけでは結論に届かない |
| 確かめられる答え | 根拠の SQL と、引用した文書（ページ）を示すので、人が確認して判断できる |
| ガバナンス | テーブルも文書も Unity Catalog で管理し、質問した人の権限で読む |

次のハンズオン2では、同じ会社のデータで、コールセンター約50人が毎日使う **回答支援エージェント** をハイコードで作り、評価・管理・デプロイ・監視（AgentOps）までを Databricks の中で行います。